In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc

In [ ]:
hdu_CG_RM = fits.open('/home2/DATA_AO/CGPS_GMIMS/CG_RM_AO_conv_2.fits')
CG_RM = hdu_CG_RM[0].data
print(CG_RM.shape)

hdu_G_RM = fits.open('/home2/DATA_AO/CGPS_GMIMS/G_RM_AO.fits')
G_RM = hdu_G_RM[0].data
print(G_RM.shape)

hdu_C_RM = fits.open('/home2/DATA_AO/CGPS_GMIMS/C_RM_AO_conv_2.fits')
C_RM = hdu_C_RM[0].data
print(C_RM.shape)

CG_PI_list = []
G_PI_list = []
C_PI_list = []

band = ['A','B','C','D']
bandlc = ['a','b','c','d']
for i in range(0,4):
    
    hdu_CG_PI = fits.open('/home2/DATA_AO/CGPS_GMIMS/PI'+band[i]+'_CG_conv_2.fits')
    hdu_G_PI = fits.open('/home2/DATA_AO/CGPS_GMIMS/PI'+band[i]+'_G.fits')
    hdu_C_PI = fits.open('/home2/DATA_AO/CGPS_GMIMS/PI'+band[i]+'_C.fits')
    CG_PI_list.append(hdu_CG_PI[0].data)
    G_PI_list.append(hdu_G_PI[0].data)
    C_PI_list.append(hdu_C_PI[0].data)
    

In [ ]:
G_PI = np.nanmean(np.array(G_PI_list),axis=0)
C_PI = np.nanmean(np.array(C_PI_list),axis=0)
CG_PI = np.nanmean(np.array(CG_PI_list),axis=0)

In [ ]:
del CG_PI_list, G_PI_list,C_PI_list
gc.collect()

In [ ]:
PI_lim = 0.1

wGbad = np.where(G_PI < PI_lim)
wCGbad = np.where(CG_PI < PI_lim)

CG_RM_good = CG_RM.copy()
CG_RM_good[wCGbad] = np.nan

G_RM_good = G_RM.copy()
G_RM_good[wGbad] = np.nan

del wGbad,wCGbad
gc.collect()

In [ ]:
fs = 18
fig = plt.figure(figsize=(20, 16))

PIlim = 0.4
RMlim = 400

c = SkyCoord([100,95], [-1,3], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(331,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(332,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(333,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(334,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(335,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(336,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(337,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(339,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')